## Objective
Given a certain stack depth and blind structure, the code aims to solve for the optimal strategy when playing Texas No-limit Hold'em (All-in or Fold variant) against one other player, with or without 27 bounty


## Setup
Both players's strategies are initialized to all-in any 2 cards. Each player will always play in constant position, and use neural network to adjust their strategy, using **L1 EV Loss** as the loss function
In some sense, it will look like some sort of Generative Adversial Network, except it generates **preflop range charts** instead of multimedia data


## Training
Both models will calculate their **EV loss** when using their current strategy against the **average of opponent's past 10 strategy charts** in each epoch


## Visualization
I hope that the learnt frequency after training can be visualized to actual poker range matrix where pocket pairs form a diagnoal with AA at top left and 22 at bottom right, suited cards at top right triangle and offsuit cards at bottom left triangle

In [2]:
# import section
from typing import Tuple, List
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
# solver settings, SB and BB chips better be set as multiples of 8 or 16 to enable bitshift operations
# nvm, pytorch wouldnt like ints
DEPTH_BB = 8
ANTE_BB = 1
NUM_EPOCHS = 1000
NUM_HANDS = 5000
MLP_DEPTH = [256,512,256]
LEARNING_RATE = 0.001
SB_VALUE = 1
BB_VALUE = 2
SEED = 30624770
BOUNTY_27 = 0
    
### Do NOT Touch below ###
STACK_SIZE = DEPTH_BB*BB_VALUE
ANTE = ANTE_BB*BB_VALUE
POT_SIZE = STACK_SIZE*2 + ANTE
BOUNTY_POT_SIZE = POT_SIZE + BOUNTY_27
### Do NOT touch above ###

# Reproducibility
if SEED != -1:
    np.random.seed(SEED)
    torch.manual_seed(SEED)

## Model definition

We will use MLP as the model to train, evaluate and test player strategies, because that is the only thing I learnt in the Grand Scheme of HKUST

### Input Layer
A flattened 13*13 vector denoting the average of past 10 strategies that opponent outputted

### Output Layer
A flattened 13*13 vector denoting the [exploitative response](https://redchippoker.com/exploitative-poker-101/) learnt by the player

In [4]:

# Architecture definition
class PlayerMLP(nn.Module):
    """
    Simple MLP representing a player (SB or BB) for All-in-or-Fold preflop decisions.
    Output: probability of choosing All-in (via softmax).
    """
    def __init__(self, name: str, hidden_layers: List[int] = [128]):
        super().__init__()
        self.name = name
        self.hidden_layers = hidden_layers
        self.ReLU = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        self.layers = nn.ModuleList()
        self.strat_history: torch.Tensor = torch.ones((1, 169), dtype=torch.float32, device=device)  # shape (num_hands, 169)
        # first layer
        self.layers.append(nn.Linear(169, hidden_layers[0]))
        self.layers.append(self.ReLU)
        # intermediate hidden layers
        for i in range(len(hidden_layers)-1):
            self.layers.append(nn.Linear(hidden_layers[i], hidden_layers[i+1]))
            self.layers.append(self.ReLU)
            # last layer
        self.layers.append(nn.Linear(hidden_layers[-1], 169))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: shape (batch_size, input_dim)
        
        returns: shape (batch_size, 169) with probabilities
        """
        for layer in self.layers:
            x = layer(x)
        return self.sigmoid(x)
    
# Instantiate model
smallBlind = PlayerMLP("Small Blind")
smallBlind.to(device)
bigBlind = PlayerMLP("Big Blind")
bigBlind.to(device)

PlayerMLP(
  (ReLU): ReLU()
  (sigmoid): Sigmoid()
  (layers): ModuleList(
    (0): Linear(in_features=169, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=169, bias=True)
  )
)

In [ ]:
# "data"loader and helper functions
NP_MATRIX: np.ndarray = np.fromfile(r".\PokerLookupTable\equity_169x169_matrix.bin", dtype=np.float32)
EQUITY_MATRIX: torch.Tensor = torch.from_numpy(NP_MATRIX).to(device)

def get_rolling_10_strats(player: PlayerMLP)->torch.Tensor:
    if player.strat_history.shape[0] < 10:
        # players are guaranteed to have at least the starting strategy of going all-in i.e. torch.ones((1, 169))
        return player.strat_history.mean(dim=0, keepdim=True);
    return player.strat_history[-10:].mean(dim=0, keepdim=True);

FREQUENCY: list[int] = [ 6,  4,  4,  4,  4,  4,  4,  4,  4,  4,  4,  4, 4,
                        12,  6,  4,  4,  4,  4,  4,  4,  4,  4,  4,  4, 4,
                        12, 12,  6,  4,  4,  4,  4,  4,  4,  4,  4,  4, 4,
                        12, 12, 12,  6,  4,  4,  4,  4,  4,  4,  4,  4, 4,
                        12, 12, 12, 12,  6,  4,  4,  4,  4,  4,  4,  4, 4,
                        12, 12, 12, 12, 12,  6,  4,  4,  4,  4,  4,  4, 4,
                        12, 12, 12, 12, 12, 12,  6,  4,  4,  4,  4,  4, 4,
                        12, 12, 12, 12, 12, 12, 12,  6,  4,  4,  4,  4, 4,
                        12, 12, 12, 12, 12, 12, 12, 12,  6,  4,  4,  4, 4,
                        12, 12, 12, 12, 12, 12, 12, 12, 12,  6,  4,  4, 4,
                        12, 12, 12, 12, 12, 12, 12, 12, 12, 12,  6,  4, 4,
                        12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12,  6, 4,
                        12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 6]
BASE_FREQUENCY: torch.Tensor = torch.tensor(FREQUENCY, dtype=torch.float32, device=device)
def get_frequency_matrix(hand_idx: int)->torch.Tensor:
    actual_frequency: torch.Tensor = BASE_FREQUENCY.clone().reshape(13, 13)
    row: int = hand_idx//13
    col: int = hand_idx%13
    if (row==col): # pocket pairs, also it feels so good to unlock modulus
        actual_frequency[row, col] = 4 # pocket pair requires further operations to be tuned down to 1
        actual_frequency[row, :] /= 2; # blocks half of the combos in the same row
        actual_frequency[:, col] /= 2; # blocks half of the combos in the same column
        return actual_frequency.flatten()
    else:
        suited_row, suited_col = (row, col) if row < col else (col, row)
        offsuit_row, offsuit_col = (row, col) if row > col else (col, row)
        actual_frequency[row, col] = 256 # suited counterpart requires further operations to be tuned down to 2/3
        actual_frequency[col, row] = 256 # offsuit counterpart requires further operations to be tuned down to 6/7
        actual_frequency[row, row], actual_frequency[col, col] = 256, 256 # involved pocket paris requires further actions to be tuned down to 3
        actual_frequency[row, :] /= 4;
        actual_frequency[row, :] *= 3;
        actual_frequency[:, col] /= 4;
        actual_frequency[:, col] *= 3; # blocks 1/4 of the combos in the larger rank
        actual_frequency[col, :] /= 4;
        actual_frequency[col, :] *= 3;
        actual_frequency[:, row] /= 4;
        actual_frequency[:, row] *= 3; # blocks 1/4 of the combos in the smaller rank
        actual_frequency[suited_row, suited_col] = 3 if row<col else 2; # suited counterpart requires further operations to be tuned down to 2/3
        actual_frequency[offsuit_row, offsuit_col] = 6 if row<col else 7; # offsuit counterpart requires further operations to be tuned down to 6/7
        actual_frequency[row, row], actual_frequency[col, col] = 3,3;
        return actual_frequency.flatten()

"""
# test the frequency matrix generation
test_hand_idx = 163 #27o
test_frequency_matrix = get_frequency_matrix(test_hand_idx).reshape(13, 13)
print(f"After consiering blocker effect, there are {test_frequency_matrix.sum()} combos in total")
print(test_frequency_matrix)
"""

def get_best_response(opponent_strat: torch.Tensor)->torch.Tensor:
    """
    opponent_strat: shape (169), dtpye float32 [0,1]
    
    returns: shape (169), dtype float32 [0,1] with best response strategy
    """
    # compute best response strategy
    best_response = torch.zeros_like(opponent_strat, device=device)  # shape (batch_size, 169)
    for hand_idx in range(169):
        # get the opponent frequency matrix after considering the blocker effect
        opponent_frequency_matrix = get_frequency_matrix(hand_idx)  # shape (169,)
        
    
    return best_response
    
def get_ev_diff(cur_strat: torch.Tensor, best_response: torch.Tensor)->torch.Tensor:
    """
    cur_strat: shape (batch_size, 169)
    best_response: shape (batch_size, 169)
    
    returns: shape (batch_size,) with expected value loss
    """
    # compute expected value loss
    ev_loss = torch.sum(cur_strat * best_response * EQUITY_MATRIX, dim=1)  # shape (batch_size,)
    return ev_loss

cuda:0


In [6]:
# Loss function
class EVLoss(nn.Module):
    def __init__(self):
        super(EVLoss, self).__init__()
    
    def forward(self, actions, best, ev_diff):
        loss = nn.BCELoss()(actions, best) * abs(ev_diff)
        return loss

In [7]:
# training setup
criterion = EVLoss()
optimizer_sb = optim.Adam(smallBlind.parameters(), lr=LEARNING_RATE)
optimizer_bb = optim.Adam(bigBlind.parameters(), lr=LEARNING_RATE)

# Training loop
for epoch in range(NUM_EPOCHS):
    sb_X = get_rolling_10_strats(bigBlind)
    bb_X = get_rolling_10_strats(smallBlind)
    sb_best_response = get_best_response(sb_X)
    bb_best_response = get_best_response(bb_X)
    sb_diff = get_ev_diff(sb_X, sb_best_response)
    bb_diff = get_ev_diff(bb_X, bb_best_response)
    print("Starting training...")
    # zeroing gradients
    optimizer_sb.zero_grad()
    optimizer_bb.zero_grad()
    
    # Forward pass: both players reveal decisions simultaneously
    sb_response = smallBlind(sb_X)
    bb_response = bigBlind(bb_X)
    
    # Compute loss
    sb_loss = criterion(sb_response, sb_best_response, sb_diff)
    bb_loss = criterion(bb_response, bb_best_response, bb_diff)

    # Backward pass and optimization
    sb_loss.backward()
    bb_loss.backward()
    optimizer_sb.step()
    optimizer_bb.step()
    
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}, SB Loss: {sb_loss.item():.4f}, BB Loss: {bb_loss.item():.4f}")

RuntimeError: The size of tensor a (169) must match the size of tensor b (28561) at non-singleton dimension 1

In [ ]:
# Saving the models
sb_path = "./solutions/SB.pth"
bb_path = "./solutions/BB.pth"

# Saving small blind model
torch.save(smallBlind.state_dict(), sb_path)
print(f"Small Blind model saved to {sb_path}")

# Saving big blind model
torch.save(bigBlind.state_dict(), bb_path)
print(f"Big Blind model saved to {bb_path}")

In [ ]:
# Result visualization
import matplotlib.pyplot as plt
# Example: Plotting the action matrix for Small Blind
smallBlind.to('cpu')
sb_strategy = torch.zeros((13,13))
for rank1 in range(13):
    for rank2 in range(13):
        one_hot_hand = torch.zeros(169, dtype=torch.float32)
        one_hot_hand[rank1*13+rank2] = 1
        with torch.no_grad():
            sb_strategy[rank1][rank2] = smallBlind(one_hot_hand).item()
            
# visualize the strategy matrix
# the top left is AA, top right is A2s, bottom left is A2o, bottom right is 22
# generate a correct figure with labels and color bar
# it should be 13x13 squares, with card labels on the top left of each square
# the frequency should be represented by color, where red means allin and blue means fold
plt.figure(figsize=(8, 8))
plt.imshow(sb_strategy, cmap='coolwarm', vmin=0, vmax=1)
plt.colorbar(label='All-in Probability')
plt.xticks(ticks=np.arange(13), labels=['A','K','Q','J','T','9','8','7','6','5','4','3','2'])
plt.yticks(ticks=np.arange(13), labels=['A','K','Q','J','T','9','8','7','6','5','4','3','2'])
plt.xlabel('Second Card')
plt.ylabel('First Card')
plt.title('Small Blind All-in Strategy Matrix')
plt.show()

# same for big blind
bigBlind.to('cpu')
bb_strategy = torch.zeros((13,13))
for rank1 in range(13):
    for rank2 in range(13):
        one_hot_hand = torch.zeros(169, dtype=torch.float32)
        one_hot_hand[rank1*13+rank2] = 1
        with torch.no_grad():
            bb_strategy[rank1][rank2] = bigBlind(one_hot_hand).item()

# visualize the strategy matrix
plt.figure(figsize=(8, 8))
plt.imshow(bb_strategy, cmap='coolwarm', vmin=0, vmax=1)
plt.colorbar(label='All-in Probability')
plt.xticks(ticks=np.arange(13), labels=['A','K','Q','J','T','9','8','7','6','5','4','3','2'])
plt.yticks(ticks=np.arange(13), labels=['A','K','Q','J','T','9','8','7','6','5','4','3','2'])
plt.xlabel('Second Card')
plt.ylabel('First Card')
plt.title('Big Blind All-in Strategy Matrix')
plt.show()